In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# India CPI forecast training

This Colab notebook mounts Google Drive, loads the frozen snapshot, creates explicitly labeled synthetic training scenarios, and evaluates models only on real CPI outcomes. Synthetic data cannot create new independent evidence or guarantee better forecasts. It compares macro-feature models and baselines at 1-, 3-, and 6-month horizons.

## Find and load the uploaded snapshot

The Drive screenshot shows the snapshot folder in My Drive. This cell searches only top-level folders whose names identify the project/snapshot, then prints the exact file it selects. If the search does not find the upload, set `DATA_PATH` manually to its Drive path.

In [12]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

DRIVE_ROOT = Path('/content/drive/MyDrive')
assert DRIVE_ROOT.exists(), f'Google Drive is not mounted at {DRIVE_ROOT}'

candidate_roots = [
    p for p in DRIVE_ROOT.iterdir()
    if p.is_dir() and (
        'india_macro_dataset' in p.name.lower()
        or 'agentic-monetary-policy-india' in p.name.lower()
    )
]

parquet_candidates = []
csv_candidates = []
for root in candidate_roots:
    parquet_candidates.extend(root.rglob('base_monthly.parquet'))
    csv_candidates.extend(root.rglob('macro_monthly_available.csv'))

# Prefer the frozen 164-month modeling file, then fall back to the snapshot's sparse monthly CSV.
if parquet_candidates:
    DATA_PATH = sorted(parquet_candidates, key=lambda p: ('/modeling/v1/' not in p.as_posix().lower(), len(p.parts)))[0]
    DATA_FORMAT = 'parquet'
elif csv_candidates:
    DATA_PATH = sorted(csv_candidates, key=lambda p: len(p.parts))[0]
    DATA_FORMAT = 'csv'
else:
    print('Top-level My Drive folders:')
    print([p.name for p in DRIVE_ROOT.iterdir() if p.is_dir()])
    raise FileNotFoundError(
        'Could not find base_monthly.parquet or macro_monthly_available.csv. '
        'Set DATA_PATH manually to the uploaded dataset file.'
    )

print('Selected dataset:', DATA_PATH)
print('Format:', DATA_FORMAT)
df = pd.read_parquet(DATA_PATH) if DATA_FORMAT == 'parquet' else pd.read_csv(DATA_PATH)
print('Raw shape:', df.shape)
print('Columns:', df.columns.tolist())
display(df.head())

Selected dataset: /content/drive/MyDrive/india_macro_dataset_snapshot (1)/data/processed/macro_monthly_available.csv
Format: csv
Raw shape: (1510, 43)
Columns: ['obs_date', 'brent_price', 'cfpi_index_2012', 'cfpi_rural_index_2012', 'cfpi_rural_yoy_2012', 'cfpi_urban_index_2012', 'cfpi_urban_yoy_2012', 'cfpi_yoy_2012', 'cpi_index_2024', 'cpi_rural_index_2024', 'cpi_rural_yoy_2024', 'cpi_urban_index_2024', 'cpi_urban_yoy_2024', 'cpi_yoy_2024', 'iip_general_growth_2011_12', 'iip_general_growth_2022_23', 'iip_general_index_2011_12', 'iip_general_index_2022_23', 'rainfall_actual', 'us_cpi_index', 'us_policy_rate', 'usd_inr_monthly_average', 'us_10y_yield_monthly_average', 'us_2y_yield_monthly_average', 'us_3m_yield_monthly_average', 'bank_rate', 'repo_rate', 'reverse_repo_rate', 'sdf_rate', 'msf_rate', 'crr', 'slr', 'oil_yoy', 'us_cpi_yoy', 'cpi_yoy_2024_derived', 'cpi_rural_yoy_2024_derived', 'cpi_urban_yoy_2024_derived', 'us_yield_curve_10y_minus_2y', 'repo_change_1m', 'repo_change_3m', '

,obs_date,brent_price,cfpi_index_2012,cfpi_rural_index_2012,cfpi_rural_yoy_2012,cfpi_urban_index_2012,cfpi_urban_yoy_2012,cfpi_yoy_2012,cpi_index_2024,cpi_rural_index_2024,cpi_rural_yoy_2024,cpi_urban_index_2024,cpi_urban_yoy_2024,cpi_yoy_2024,iip_general_growth_2011_12,iip_general_growth_2022_23,iip_general_index_2011_12,iip_general_index_2022_23,rainfall_actual,us_cpi_index,us_policy_rate,usd_inr_monthly_average,us_10y_yield_monthly_average,us_2y_yield_monthly_average,us_3m_yield_monthly_average,bank_rate,repo_rate,reverse_repo_rate,sdf_rate,msf_rate,crr,slr,oil_yoy,us_cpi_yoy,cpi_yoy_2024_derived,cpi_rural_yoy_2024_derived,cpi_urban_yoy_2024_derived,us_yield_curve_10y_minus_2y,repo_change_1m,repo_change_3m,month_sin,month_cos,rainfall_anomaly_1991_2020_mm
0,1901-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.500000,8.660254e-01,16.533333
1,1901-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.866025,5.000000e-01,14.793333
2,1901-03-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,6.123234e-17,-10.330000
3,1901-04-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.866025,-5.000000e-01,-0.130000
4,1901-05-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,52.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.500000,-8.660254e-01,-7.886667


## Validate the CPI target and build the inflation series

The target column is identified explicitly as `cpi_index_2024`. No predictor imputation is performed. CPI year-over-year inflation is calculated from the CPI index and is therefore available after its first 12 months.

In [13]:
DATE_COL = 'obs_date' if 'obs_date' in df.columns else 'date'
TARGET_COL = 'cpi_index_2024'
assert DATE_COL in df.columns, f'Expected date column not found. Columns: {df.columns.tolist()}'
assert TARGET_COL in df.columns, f'Expected CPI target not found. Columns: {df.columns.tolist()}'

df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='raise')
df = df.sort_values(DATE_COL).drop_duplicates(DATE_COL).set_index(DATE_COL)
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors='raise')
df = df.loc[df[TARGET_COL].notna()].copy()

expected_dates = pd.date_range(df.index.min(), df.index.max(), freq='MS')
assert df.index.is_unique and df.index.is_monotonic_increasing
assert df.index.equals(expected_dates), 'CPI dates must be unique, ordered, consecutive month starts.'
assert len(df) == 164, f'Expected 164 CPI months, got {len(df)}'
assert df[TARGET_COL].notna().all()

cpi_index = df[TARGET_COL].astype(float)
inflation = (100 * cpi_index.pct_change(12)).dropna().rename('cpi_yoy_pp')
assert len(inflation) == 152

print(f'CPI target: {TARGET_COL}')
print(f'CPI index range: {cpi_index.index.min():%Y-%m} to {cpi_index.index.max():%Y-%m} ({len(cpi_index)} observations)')
print(f'Derived YoY inflation range: {inflation.index.min():%Y-%m} to {inflation.index.max():%Y-%m} ({len(inflation)} observations)')
print('Predictor values are not imputed.')
display(pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum()}))

CPI target: cpi_index_2024
CPI index range: 2013-01 to 2026-08 (164 observations)
Derived YoY inflation range: 2014-01 to 2026-08 (152 observations)
Predictor values are not imputed.


,dtype,missing
brent_price,float64,0
cfpi_index_2012,float64,8
cfpi_rural_index_2012,float64,8
cfpi_rural_yoy_2012,float64,21
cfpi_urban_index_2012,float64,8
cfpi_urban_yoy_2012,float64,21
cfpi_yoy_2012,float64,21
cpi_index_2024,float64,0
cpi_rural_index_2024,float64,0
cpi_rural_yoy_2024,float64,24


## Define models and macro features

Models use CPI YoY lags, one-month-lagged numeric macro categories, and forecast-month seasonality. Missing predictors are imputed inside each model pipeline; CPI targets are never imputed. Synthetic augmentation is evaluated separately.

In [14]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

MODEL_NAMES = ['ridge', 'random_forest', 'hist_gradient_boosting']
try:
    from xgboost import XGBRegressor
    MODEL_NAMES.append('xgboost')
except ImportError:
    XGBRegressor = None

LAGS = (1, 2, 3, 6, 12)
HORIZONS = (1, 3, 6)
MIN_TRAIN_MONTHS = 72
EXCLUDED_PREDICTORS = {TARGET_COL}
PREDICTOR_COLUMNS = [c for c in df.columns if c not in EXCLUDED_PREDICTORS and pd.api.types.is_numeric_dtype(df[c])]
macro_features = df[PREDICTOR_COLUMNS].apply(pd.to_numeric, errors='coerce').shift(1).reindex(inflation.index)


def calendar_terms(date):
    angle = 2 * np.pi * (date.month - 1) / 12
    return [np.sin(angle), np.cos(angle)]


def make_features(values, dates, macro_matrix, origin, horizon):
    target_date = dates[origin + horizon]
    macro_row = macro_matrix.iloc[origin].to_numpy(dtype=float)
    return np.asarray([values[origin-lag+1] for lag in LAGS] + macro_row.tolist() + calendar_terms(target_date), dtype=float)


def make_training_data(values, dates, macro_matrix, cutoff, horizon):
    rows, labels = [], []
    for origin in range(max(LAGS)-1, cutoff-horizon+1):
        rows.append(make_features(values, dates, macro_matrix, origin, horizon))
        labels.append(values[origin+horizon])
    return np.asarray(rows), np.asarray(labels)


def make_model(name):
    if name == 'ridge':
        estimator, scale = Ridge(alpha=10.0), True
    elif name == 'random_forest':
        estimator, scale = RandomForestRegressor(n_estimators=150, max_depth=3, min_samples_leaf=6, max_features=1.0, random_state=17, n_jobs=1), False
    elif name == 'hist_gradient_boosting':
        estimator, scale = HistGradientBoostingRegressor(max_iter=80, learning_rate=0.03, max_leaf_nodes=3, min_samples_leaf=10, l2_regularization=10.0, early_stopping=False, random_state=17), False
    elif name == 'xgboost' and XGBRegressor is not None:
        estimator, scale = XGBRegressor(objective='reg:squarederror', n_estimators=80, max_depth=1, learning_rate=0.03, min_child_weight=5, reg_lambda=20.0, subsample=1.0, colsample_bytree=1.0, n_jobs=1, random_state=17, verbosity=0), False
    else:
        raise ValueError(f'Unknown model: {name}')
    steps = [('imputer', SimpleImputer(strategy='median', add_indicator=True))]
    if scale:
        steps.append(('scaler', StandardScaler()))
    steps.append(('model', estimator))
    return Pipeline(steps)

print('Macro predictors:', PREDICTOR_COLUMNS)
print('Models:', ['persistence', 'seasonal_naive'] + MODEL_NAMES)
print('Horizons:', HORIZONS)


Macro predictors: ['brent_price', 'cfpi_index_2012', 'cfpi_rural_index_2012', 'cfpi_rural_yoy_2012', 'cfpi_urban_index_2012', 'cfpi_urban_yoy_2012', 'cfpi_yoy_2012', 'cpi_rural_index_2024', 'cpi_rural_yoy_2024', 'cpi_urban_index_2024', 'cpi_urban_yoy_2024', 'cpi_yoy_2024', 'iip_general_growth_2011_12', 'iip_general_growth_2022_23', 'iip_general_index_2011_12', 'iip_general_index_2022_23', 'rainfall_actual', 'us_cpi_index', 'us_policy_rate', 'usd_inr_monthly_average', 'us_10y_yield_monthly_average', 'us_2y_yield_monthly_average', 'us_3m_yield_monthly_average', 'bank_rate', 'repo_rate', 'reverse_repo_rate', 'sdf_rate', 'msf_rate', 'crr', 'slr', 'oil_yoy', 'us_cpi_yoy', 'cpi_yoy_2024_derived', 'cpi_rural_yoy_2024_derived', 'cpi_urban_yoy_2024_derived', 'us_yield_curve_10y_minus_2y', 'repo_change_1m', 'repo_change_3m', 'month_sin', 'month_cos', 'rainfall_anomaly_1991_2020_mm']
Models: ['persistence', 'seasonal_naive', 'ridge', 'random_forest', 'hist_gradient_boosting', 'xgboost']
Horizons:

## Generate labeled synthetic training scenarios

The next cell creates 20 joint-row calendar-year block-bootstrap scenarios using complete historical donor years through December 2019. It preserves cross-series row relationships and original gaps. These rows are synthetic, add no independent economic evidence, and are never used for evaluation. Larger copies of the same history would not make the models more reliable.

In [15]:
import hashlib
from datetime import datetime, timezone

N_SYNTHETIC_SCENARIOS = 20
SYNTHETIC_SEED = 42
SYNTHETIC_WEIGHT_SHARE = 0.25
DONOR_END = pd.Timestamp('2019-12-01')
donor_yoy = inflation.loc[:DONOR_END]
donor_years = [int(y) for y in sorted(donor_yoy.index.year.unique()) if len(donor_yoy[donor_yoy.index.year == y]) == 12]
assert len(donor_years) >= 5, f'Expected at least five complete donor years, found {donor_years}'
rng = np.random.default_rng(SYNTHETIC_SEED)
synthetic_dates = inflation.index
synthetic_frames = []
for scenario_id in range(N_SYNTHETIC_SCENARIOS):
    chosen_years = rng.choice(donor_years, size=int(np.ceil(len(synthetic_dates)/12)), replace=True)
    donor_dates = []
    for year in chosen_years:
        donor_dates.extend(donor_yoy[donor_yoy.index.year == int(year)].index.tolist())
    donor_dates = donor_dates[:len(synthetic_dates)]
    frame = df.reindex(donor_dates).copy().reset_index()
    frame = frame.rename(columns={frame.columns[0]: 'source_donor_date'})
    frame.insert(0, 'obs_date', synthetic_dates.to_numpy())
    frame['synthetic_scenario_id'] = f'scenario_{scenario_id+1:02d}'
    frame['data_origin'] = 'synthetic_block_bootstrap'
    frame['synthetic_cpi_yoy_pp'] = inflation.reindex(donor_dates).to_numpy(dtype=float)
    cpi_values = pd.to_numeric(frame[TARGET_COL], errors='coerce').to_numpy(dtype=float)
    for t in range(12, len(cpi_values)):
        prior, yoy = cpi_values[t-12], frame.loc[t, 'synthetic_cpi_yoy_pp']
        cpi_values[t] = prior*(1+yoy/100) if np.isfinite(prior) and np.isfinite(yoy) else np.nan
    frame[TARGET_COL] = cpi_values
    synthetic_frames.append(frame)

synthetic_panel = pd.concat(synthetic_frames, ignore_index=True)
OUTPUT_DIR = DRIVE_ROOT / 'india_macro_training_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
synthetic_path = OUTPUT_DIR / 'synthetic_macro_scenarios.csv'
synthetic_panel.to_csv(synthetic_path, index=False)
source_sha256 = hashlib.sha256(Path(DATA_PATH).read_bytes()).hexdigest()
synthetic_manifest = {
    'synthetic': True, 'method': 'joint_row_calendar_year_block_bootstrap',
    'scenario_count': N_SYNTHETIC_SCENARIOS, 'rows': int(len(synthetic_panel)),
    'seed': SYNTHETIC_SEED, 'donor_start': str(donor_yoy.index.min().date()),
    'donor_end': str(donor_yoy.index.max().date()), 'source_file': str(DATA_PATH),
    'source_sha256': source_sha256, 'predictor_columns': PREDICTOR_COLUMNS,
    'warning': 'Training augmentation only; synthetic rows are not new observations and must not enter validation/test.',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
}
with (OUTPUT_DIR/'synthetic_generation_manifest.json').open('w',encoding='utf-8') as f:
    json.dump(synthetic_manifest,f,indent=2)
print('Synthetic rows:',len(synthetic_panel),'scenarios:',N_SYNTHETIC_SCENARIOS)
print('Saved:',synthetic_path,'| donor years:',donor_years)


Synthetic rows: 3040 scenarios: 20
Saved: /content/drive/MyDrive/india_macro_training_outputs/synthetic_macro_scenarios.csv | donor years: [2014, 2015, 2016, 2017, 2018, 2019]


## Rolling-origin training and real-data evaluation

Each prediction is scored against an actual CPI outcome. Synthetic rows can enter training only when their donor observations are available at that forecast origin.

In [16]:
import warnings

values = inflation.to_numpy(dtype=float)
dates = inflation.index
first_origin = MIN_TRAIN_MONTHS - 1
last_origin = len(inflation) - max(HORIZONS) - 1
forecast_rows = []

for origin in range(first_origin, last_origin+1):
    origin_date = dates[origin]
    for horizon in HORIZONS:
        target_pos = origin+horizon
        actual = float(values[target_pos])
        for name, pred in {'persistence':float(values[origin]), 'seasonal_naive':float(values[target_pos-12])}.items():
            forecast_rows.append({'origin_date':origin_date,'target_date':dates[target_pos],'horizon_months':horizon,'model':name,'actual_yoy_pp':actual,'predicted_yoy_pp':pred})
        X_real,y_real = make_training_data(values,dates,macro_features,origin,horizon)
        X_test = make_features(values,dates,macro_features,origin,horizon).reshape(1,-1)
        for model_name in MODEL_NAMES:
            real_model = make_model(model_name)
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                real_model.fit(X_real,y_real)
                pred = float(np.asarray(real_model.predict(X_test)).reshape(-1)[0])
            forecast_rows.append({'origin_date':origin_date,'target_date':dates[target_pos],'horizon_months':horizon,'model':f'real_{model_name}','actual_yoy_pp':actual,'predicted_yoy_pp':pred})

            syn_x,syn_y = [],[]
            for _,scenario in synthetic_panel.groupby('synthetic_scenario_id',sort=False):
                scenario = scenario.set_index('obs_date').sort_index()
                syn_dates = pd.DatetimeIndex(scenario.index)
                syn_values = scenario['synthetic_cpi_yoy_pp'].to_numpy(dtype=float)
                donor_dates_local = pd.to_datetime(scenario['source_donor_date']).to_numpy()
                allowed = (syn_dates <= origin_date) & (donor_dates_local <= origin_date.to_datetime64())
                syn_macro = scenario[PREDICTOR_COLUMNS].apply(pd.to_numeric,errors='coerce').shift(1)
                for si in range(max(LAGS)-1,len(syn_dates)-horizon):
                    if allowed[si+horizon]:
                        syn_x.append(make_features(syn_values,syn_dates,syn_macro,si,horizon))
                        syn_y.append(syn_values[si+horizon])
            if syn_x:
                syn_x,syn_y = np.asarray(syn_x),np.asarray(syn_y)
                fit_x,fit_y = np.vstack([X_real,syn_x]),np.concatenate([y_real,syn_y])
                weights = np.r_[np.ones(len(y_real)),np.full(len(syn_y),SYNTHETIC_WEIGHT_SHARE*len(y_real)/len(syn_y))]
                augmented = make_model(model_name)
                with warnings.catch_warnings():
                    warnings.simplefilter('ignore')
                    augmented.fit(fit_x,fit_y,model__sample_weight=weights)
                    aug_pred = float(np.asarray(augmented.predict(X_test)).reshape(-1)[0])
                forecast_rows.append({'origin_date':origin_date,'target_date':dates[target_pos],'horizon_months':horizon,'model':f'augmented_{model_name}','actual_yoy_pp':actual,'predicted_yoy_pp':aug_pred})

forecasts = pd.DataFrame(forecast_rows)
print('Rolling origins:',forecasts.origin_date.nunique(),'forecast rows:',len(forecasts))
display(forecasts.head())


Rolling origins: 75 forecast rows: 2250


,origin_date,target_date,horizon_months,model,actual_yoy_pp,predicted_yoy_pp
0,2019-12-01,2020-01-01,1,persistence,7.619048,7.317073
1,2019-12-01,2020-01-01,1,seasonal_naive,7.619048,1.941748
2,2019-12-01,2020-01-01,1,real_ridge,7.619048,6.937370
3,2019-12-01,2020-01-01,1,augmented_ridge,7.619048,6.766725
4,2019-12-01,2020-01-01,1,real_random_forest,7.619048,5.340498


## Compare models by horizon

Metrics compare all variants on the same real CPI outcomes. Compare each `augmented_*` model to its matching `real_*` version and to simple baselines. Synthetic training fit is not an evaluation metric.

In [17]:
metric_rows = []
for (horizon, model_name), part in forecasts.groupby(['horizon_months', 'model']):
    error = part.predicted_yoy_pp.to_numpy() - part.actual_yoy_pp.to_numpy()
    metric_rows.append({
        'horizon_months': int(horizon),
        'model': model_name,
        'n_forecasts': len(part),
        'MAE_pp': float(np.mean(np.abs(error))),
        'RMSE_pp': float(np.sqrt(np.mean(error ** 2))),
        'bias_pp': float(np.mean(error)),
    })

metrics = pd.DataFrame(metric_rows).sort_values(['horizon_months', 'MAE_pp'])
display(metrics.round(4))
print('Lowest MAE per horizon:')
display(metrics.groupby('horizon_months', as_index=False).first().round(4))

,horizon_months,model,n_forecasts,MAE_pp,RMSE_pp,bias_pp
4,1,persistence,75,0.6435,0.8384,0.0522
1,1,augmented_random_forest,75,0.6865,0.8931,0.0597
6,1,real_random_forest,75,0.7033,0.9273,0.0497
7,1,real_ridge,75,0.8050,1.0166,-0.0790
0,1,augmented_hist_gradient_boosting,75,0.8291,1.0556,-0.0103
2,1,augmented_ridge,75,0.8505,1.0763,0.0005
5,1,real_hist_gradient_boosting,75,0.8805,1.1068,-0.0331
3,1,augmented_xgboost,75,0.9409,1.1954,-0.1157
8,1,real_xgboost,75,0.9925,1.2461,-0.0844
9,1,seasonal_naive,75,1.9026,2.3749,0.2683


Lowest MAE per horizon:


,horizon_months,model,n_forecasts,MAE_pp,RMSE_pp,bias_pp
0,1,persistence,75,0.6435,0.8384,0.0522
1,3,persistence,75,1.1190,1.3882,0.1417
2,6,augmented_hist_gradient_boosting,75,1.2760,1.6241,0.2420


## Refit models and create a six-month research forecast

The origin is the latest CPI month in the uploaded dataset. These estimates are research output, not official CPI values. Use real rolling-origin results to guide model selection.

In [18]:
final_origin = len(inflation)-1
future_rows = []
for horizon in range(1,7):
    future_date = dates[-1]+pd.offsets.MonthBegin(horizon)
    X_train,y_train = make_training_data(values,dates,macro_features,final_origin,horizon)
    X_future = np.asarray([values[final_origin-lag+1] for lag in LAGS]+macro_features.iloc[final_origin].to_numpy(dtype=float).tolist()+calendar_terms(future_date),dtype=float).reshape(1,-1)
    for model_name in MODEL_NAMES:
        model = make_model(model_name)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            model.fit(X_train,y_train)
            prediction = float(np.asarray(model.predict(X_future)).reshape(-1)[0])
        future_rows.append({'forecast_origin':dates[-1],'forecast_date':future_date,'horizon_months':horizon,'model':f'real_{model_name}','forecast_cpi_yoy_pp':prediction})
    future_rows.extend([
        {'forecast_origin':dates[-1],'forecast_date':future_date,'horizon_months':horizon,'model':'persistence','forecast_cpi_yoy_pp':float(values[final_origin])},
        {'forecast_origin':dates[-1],'forecast_date':future_date,'horizon_months':horizon,'model':'seasonal_naive','forecast_cpi_yoy_pp':float(values[final_origin+horizon-12])},
    ])
future_forecasts = pd.DataFrame(future_rows)
display(future_forecasts.pivot(index=['forecast_date','horizon_months'],columns='model',values='forecast_cpi_yoy_pp').round(3))


,model,persistence,real_hist_gradient_boosting,real_random_forest,real_ridge,real_xgboost,seasonal_naive
forecast_date,horizon_months,,,,,,
2026-09-01,1,4.82,5.001,4.852,4.745,4.901,1.408
2026-10-01,2,4.82,4.721,4.660,4.185,4.927,0.039
2026-11-01,3,4.82,4.570,4.473,4.082,4.762,0.493
2026-12-01,4,4.82,4.445,4.404,5.381,4.549,1.166
2027-01-01,5,4.82,4.486,3.893,5.099,4.551,2.734
2027-02-01,6,4.82,4.419,4.128,5.166,4.598,3.208


## Save outputs to Google Drive

The source snapshot is read-only. Results and synthetic artifacts are written to a separate Drive output directory.

In [19]:
OUTPUT_DIR = DRIVE_ROOT/'india_macro_training_outputs'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
metrics.to_csv(OUTPUT_DIR/'rolling_origin_metrics.csv',index=False)
forecasts.to_csv(OUTPUT_DIR/'rolling_origin_forecasts.csv',index=False)
future_forecasts.to_csv(OUTPUT_DIR/'research_forecasts_next_6_months.csv',index=False)
print('Saved real evaluation, forecasts, synthetic scenarios, and manifest to:',OUTPUT_DIR)

Saved real evaluation, forecasts, synthetic scenarios, and manifest to: /content/drive/MyDrive/india_macro_training_outputs


## Notes

- Synthetic scenarios are a labeled block bootstrap and add no independent economic evidence.
- Only actual CPI outcomes are used for rolling-origin metrics; synthetic records are training-only.
- Numeric macro predictors are lagged one month and imputed within each training fold. CPI targets are never imputed.
- Keep augmentation only if it improves real rolling-origin errors consistently against the real-only model and baselines.
- The short history, predictor gaps, and unavailable publication vintages mean results are research-grade, not a production guarantee.